# [LAB-09] 6. [PBT] 회귀모형 평가와 진단 — 연습문제

## 준비작업

### 라이브러리 참조

In [1]:
from jussam import load_data
from helpers import *

import numpy as np
from pandas import DataFrame

# 성능 평가 지표 (직접 계산하지 않고 sklearn 의 함수를 사용한다)
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error

📦 연세대학교 주영아 교수가 제작한 라이브러리를 사용중입니다.
📧 Email(1): j.purplerose@yonsei.ac.kr
📧 Email(2): j.purplerose@gmail.com
📝 Website: https://juyounga.kr/


## 📚 요율 담당자의 모형 채택 고민 — 전처리를 더 하면 더 좋은 모형일까?

### 당신은 한빛손해보험의 요율 분석 담당자입니다.

지난주 EDA에서 가입자 1,337명의 연간 보험료(charges)에 영향을 주는 변수로 흡연 여부(smoker)와 나이(age)를 채택하고, 자녀 수(children)와 체질량지수(bmi)는 후보로 남겼습니다. 성별과 지역은 유의하지 않아 제외했습니다.

전처리 팀은 그 결과를 받아 **로그변환 → 이상치 대체 → 다중공선성 제거 → 스케일링**을 하나씩 누적한 5개 데이터셋(0~4번 체크포인트)을 넘겨주었습니다. 팀장은 "전처리를 가장 많이 한 4번이 당연히 제일 좋은 모형이겠지"라고 말합니다.

다섯 데이터셋으로 각각 모형을 적합해 **어느 체크포인트를 최종 요율 모형으로 채택할지** 정하고, 그 모형을 그대로 요율 산정에 써도 되는지 **진단 결과까지** 확인하세요.

### 💻 코드 작성

#### 데이터 가져오기

In [2]:
df0 = load_data("insurance_checkpoint_0")
df1 = load_data("insurance_checkpoint_1")
df2 = load_data("insurance_checkpoint_2")
df3 = load_data("insurance_checkpoint_3")
df4 = load_data("insurance_checkpoint_4")

df0.head()

📚 보험 비용 데이터셋의 전처리 체크포인트 0 - 기준선 역할을 위해 라벨링과 더미변수 처리만 적용됨
📚 보험 비용 데이터셋의 전처리 체크포인트 1 - 라벨링 + 더미변수 + 로그변환
📚 보험 비용 데이터셋의 전처리 체크포인트 2 - 라벨링 + 더미변수 + 로그변환 + 이상치 대체
📚 보험 비용 데이터셋의 전처리 체크포인트 3 - 라벨링 + 더미변수 + 로그변환 + 이상치 대체 + 다중공선성제거
📚 보험 비용 데이터셋의 전처리 체크포인트 4 - 라벨링 + 더미변수 + 로그변환 + 이상치 대체 + 다중공선성제거 + 스케일링


,age,bmi,children,smoker,charges
0,19,27.900,0,1,16884.924
1,18,33.770,1,0,1725.552
2,28,33.000,3,0,4449.462
3,33,22.705,0,0,21984.471
4,32,28.880,0,0,3866.855


#### 분석 대상 정리

In [3]:
# 종속변수
target = "charges"

# 비교 대상 체크포인트 (아래로 갈수록 전처리가 하나씩 누적된다)
checkpoints = { "0_기준선": df0, "1_로그변환": df1, "2_이상치대체": df2,
    "3_공선성제거": df3, "4_스케일링": df4 }

# 종속변수(charges)에 순수 log 변환이 적용되었는지 여부 (체크포인트 1부터 적용됨)
log_y = { "0_기준선": False, "1_로그변환": True, "2_이상치대체": True,
    "3_공선성제거": True, "4_스케일링": True }

# log1p 가 적용된 독립변수 (기준선은 없고, 체크포인트 1부터 children 에 적용됨)
log1p_x = { "0_기준선": [], "1_로그변환": ["children"], "2_이상치대체": ["children"],
    "3_공선성제거": ["children"], "4_스케일링": ["children"] }

In [ ]:
# children(자녀 수)은 count형 변수이기 때문. 

# charges(보험료)는 항상 양수라서 순수 log()로 변환해도 문제없음
# 근데 children은 0명인 사람이 있음 → log(0)은 정의가 안 됨 (-inf)

# 그래서 0을 포함한 값에도 안전하게 쓸 수 있는 log1p(x) = log(1+x)를 쓴 거야. 
# log1p(0) = log(1) = 0이라서 0값이 깨지지 않거든.

#### 보험료의 척도 확인

In [4]:
charges_scale = DataFrame()

for name in checkpoints:
    data = checkpoints[name]
    charges_scale[name] = data[target].describe()

charges_scale.round(3)

,0_기준선,1_로그변환,2_이상치대체,3_공선성제거,4_스케일링
count,1337.000,1337.000,1337.000,1337.000,1337.000
mean,13279.121,9.100,9.100,9.100,9.100
std,12110.360,0.919,0.919,0.919,0.919
min,1121.874,7.023,7.023,7.023,7.023
25%,4746.344,8.465,8.465,8.465,8.465
50%,9386.161,9.147,9.147,9.147,9.147
75%,16657.717,9.721,9.721,9.721,9.721
max,63770.428,11.063,11.063,11.063,11.063


#### 체크포인트별 모형 적합 및 성능 비교

In [5]:
fits = {}
result = []

for c in checkpoints:
    data = checkpoints[c]       # 체크포인트 데이터프레임
    log = log_y[c]              # 종속변수에 순수 log 가 적용되었는지 여부

    # --- 1) 모형 적합 (유의하지 않은 변수는 후진소거로 제거) ---
    fit = my_ols.auto_ols(data, target,
                log_y=log, log1p_x=log1p_x[c],
                backward=True, report=False, test=False)

    fits[c] = fit

    # --- 2) 실제값과 예측값을 원본 척도(달러)로 되돌린다 ---
    if log:
        # 종속변수는 순수 log 변환이므로 역변환은 exp 를 사용한다 (log1p 면 expm1)
        y_true = np.exp(data[target])
        y_pred = np.exp(fit.fittedvalues)
    else:
        y_true = data[target]       # 체크포인트 0 은 이미 원본 척도
        y_pred = fit.fittedvalues

    # --- 3) 지표를 한 행으로 정리해 저장 ---
    result.append({
        "모델": c,
        "변수수": int(fit.df_model),        # 상수항을 제외한 독립변수 개수
        "R2(모델척도)": fit.rsquared,       # 척도가 다르면 비교 불가 (참고용)
        "Adj.R2": fit.rsquared_adj,
        "AIC": fit.aic,
        # 아래 세 지표는 원본 척도(달러)에서 계산하므로 체크포인트간 비교가 가능하다
        "R2(원본척도)": r2_score(y_true, y_pred),
        "RMSE": root_mean_squared_error(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
    })

rdf = DataFrame(result).set_index("모델")
rdf.round(3)

,변수수,R2(모델척도),Adj.R2,AIC,R2(원본척도),RMSE,MAE
모델,,,,,,,
0_기준선,4,0.750,0.749,27092.752,0.750,6058.642,4181.335
1_로그변환,4,0.763,0.762,1653.311,0.532,8281.268,4247.790
2_이상치대체,4,0.763,0.762,1652.808,0.532,8278.466,4248.574
3_공선성제거,4,0.763,0.762,1652.808,0.532,8278.466,4248.574
4_스케일링,4,0.763,0.762,1652.808,0.532,8278.466,4248.574


#### 최종 모형의 회귀계수와 영향력 순위

In [6]:
final_name = "0_기준선"

final_fit = fits[final_name]
final_data = checkpoints[final_name]

print(f"최종 모형: {final_name} | 등분산 위배(HC3 사용) = {final_fit.use_hc3_}")

my_ols.report_variables(final_fit, final_data, hc3=final_fit.use_hc3_)

최종 모형: 0_기준선 | 등분산 위배(HC3 사용) = True


,종속변수,독립변수,B,표준오차,표준오차(HC3),β,t,t(HC3),유의확률,유의확률(HC3),공차,VIF
0,charges,smoker,23810.399,411.414,573.837,0.794,57.875,41.493,0.000,0.000,0.999,1.001
1,charges,age,257.773,11.910,11.877,0.299,21.644,21.704,0.000,0.000,0.986,1.014
2,charges,bmi,321.871,27.388,30.077,0.162,11.752,10.702,0.000,0.000,0.988,1.012
3,charges,children,472.975,137.879,131.271,0.047,3.430,3.603,0.001,0.000,0.998,1.002


#### 최종 모형의 회귀분석 가정 검정

In [7]:
my_ols.test_linear(final_fit, plot=False)       # 1) 선형성
my_ols.test_normal(final_fit, plot=False)      # 2) 정규성
my_ols.test_equalvar(final_fit)                # 3) 등분산성
my_ols.test_independent(final_fit)             # 4) 독립성

,statistic,p-value,linearity,result
Ramsey RESET,141.672,0.000,False,대립가설 채택 → 선형성 위배(곡선 관계 존재)


,statistic,p-value,normality,result
Kolmogorov-Smirnov,0.163,0.000,False,대립가설 채택 → 정규성 위배


,기대(%),허용범위(%),실제(%),판정
구간,,,,
±1√MSE,68.000,65~71,73.000,위배
±2√MSE,95.000,94~96,94.690,충족
±3√MSE,99.700,99~100,97.980,위배


√MSE = 6070.00 · 구간 규칙 판정: 정규성 위배


,LM statistic,LM p-value,F statistic,F p-value,homoscedasticity,result
Breusch-Pagan,116.980,0.000,31.929,0.000,False,대립가설 채택 → 등분산 아님


,statistic,independence,result
Durbin-Watson,2.088,True,독립성 만족


### 문제 풀이

#### 1. 전처리 팀이 넘긴 다섯 데이터셋의 보험료 분포를 나란히 놓고 비교했을 때, 보험료가 원래의 달러 금액이 아닌 다른 척도로 바뀌어 있는 데이터셋은 몇 개인가요?

- **정답**: `4`
- **복습 개념**: 종속변수의 척도 확인입니다. 전처리 단계마다 종속변수가 그대로 남아 있는지 변환되었는지를 먼저 확인해야, 뒤에 나오는 성능지표를 서로 비교할 수 있는지 판단할 수 있습니다.
- **풀이 접근**: 다섯 데이터셋의 보험료 열에 대해 기술통계(개수·평균·최솟값·최댓값)를 각각 구해 한 표에 나란히 붙여 놓고, 값의 크기가 어디서부터 달라지는지 봅니다.
- **근거(계산 결과)**: 0번 체크포인트만 평균 13,279.121 · 최댓값 63,770.428 로 달러 금액 그대로이고, 1~4번은 평균 9.100 · 최댓값 11.063 입니다. 즉 **4개**가 로그 척도로 바뀌어 있습니다. 로그변환이 1번 단계에서 적용된 뒤 2·3·4번으로 그대로 누적되었기 때문입니다.
- **실무 포인트**: 종속변수의 척도가 섞여 있으면 R²·AIC·BIC 처럼 종속변수의 분산에 기대는 지표는 데이터셋 사이에서 비교할 수 없습니다. 성능표를 읽기 전에 이 확인을 먼저 하는 습관을 들이세요.

#### 2. 다섯 모형의 성능표에서 모델 척도의 설명력(R²)만 놓고 보면 기준선보다 좋아 보이는 모형들이 있습니다. 그 중 가장 높은 설명력은 얼마인가요? (소수 셋째 자리)

- **정답**: `0.763`
- **복습 개념**: 모델 척도 R² 의 함정입니다. R² 는 "그 모형이 자기 종속변수의 분산을 얼마나 설명했는가"이므로, 종속변수가 로그로 바뀌면 애초에 설명해야 할 분산 자체가 달라집니다.
- **풀이 접근**: 다섯 데이터셋마다 후진소거를 포함해 모형을 적합하고, 각 결과의 R² 를 한 표로 모아 가장 큰 값을 읽습니다.
- **근거(계산 결과)**: R²(모델척도)는 0번 0.750, 1~4번 **0.763** 입니다. 숫자만 보면 로그변환을 한 쪽이 1.3%p 더 설명한 것처럼 보이고, AIC 도 27,092.752 대 1,652.808 로 로그 쪽이 압도적으로 작아 보입니다.
- **자주 하는 실수**: AIC 가 27,092 에서 1,652 로 줄어든 것을 "모형이 훨씬 좋아졌다"고 읽으면 안 됩니다. 종속변수의 단위가 달러에서 로그로 바뀌면서 우도 자체의 스케일이 달라진 결과일 뿐, 같은 척도가 아니면 AIC·BIC 비교는 의미가 없습니다.

#### 3. 그 0.763 짜리 모형들의 예측값을 원래의 달러 금액으로 되돌린 뒤 설명력을 다시 계산하면 얼마가 되나요? (소수 셋째 자리)

- **정답**: `0.532`
- **복습 개념**: 역변환(retransformation) 후의 성능 재평가입니다. 로그 척도에서 잘 맞는 모형이 원래 금액 단위에서도 잘 맞는다는 보장은 없습니다.
- **풀이 접근**: 로그가 적용된 데이터셋은 실제값과 예측값을 모두 지수함수로 되돌린 다음, 그 달러 값끼리 설명력·평균오차를 다시 계산합니다. 되돌리는 식은 변환식과 짝이 맞아야 합니다(순수 로그 → 지수, log1p → expm1).
- **근거(계산 결과)**: R²(원본척도)가 1~4번 모두 **0.532** 로 떨어집니다. 반면 0번은 0.750 그대로입니다. 로그 척도에서 줄인 오차는 "비율의 오차"이고, 되돌리면 보험료가 큰 고액 계약자에서 오차가 크게 증폭되기 때문입니다.
- **헷갈리기 쉬운 점**: 0.763 과 0.532 는 같은 모형의 성능입니다. 지표가 두 개 있는 게 아니라 **어느 척도에서 재느냐**가 다를 뿐입니다. 요율은 달러로 청구하므로 우리가 봐야 할 값은 0.532 쪽입니다.

#### 4. 실제 청구 단위인 달러로 되돌려 평균적인 예측 오차(RMSE)를 비교했을 때, 최종 요율 모형으로 채택해야 할 데이터셋은 어느 체크포인트인가요?

- **정답**: `0_기준선`
- **복습 개념**: 비교 가능한 지표로 모형을 선택하는 절차입니다. 척도에 좌우되지 않는 원본 척도 RMSE 를 1순위 기준으로 삼습니다.
- **풀이 접근**: 모든 모형의 실제값·예측값을 달러로 되돌려 RMSE 를 계산하고, 가장 작은 값을 가진 체크포인트를 고릅니다. 오차가 비슷하다면 변수 수가 적은 쪽(간명성)을 택합니다.
- **근거(계산 결과)**: RMSE 는 0번 **6,058.642** 달러, 1번 8,281.268, 2~4번 8,278.466 달러입니다. MAE 도 0번이 4,181.335 로 가장 작습니다. 전처리를 가장 많이 한 4번이 아니라, 라벨링과 더미변수 처리만 한 **0_기준선**이 채택됩니다. 변수 수는 다섯 모형 모두 4개로 같아 간명성으로는 갈리지 않습니다.
- **함께 생각해 볼 점**: "전처리를 많이 하면 성능이 올라간다"는 팀장의 기대는 이 데이터에서 성립하지 않았습니다. 2·3·4번의 지표가 소수점까지 똑같은 것도 눈여겨보세요. 다중공선성 제거에서 빠진 변수가 없고, 스케일링은 회귀계수의 단위만 바꾸므로 적합도가 달라지지 않습니다.

#### 5. 채택한 모형에서 보험료에 미치는 영향력이 가장 큰 변수는 무엇인가요?

- **정답**: `smoker`
- **복습 개념**: 표준화 회귀계수(β)로 읽는 영향력 순위입니다. 단위가 서로 다른 변수들을 같은 잣대로 비교하려면 원래 계수(B)가 아니라 β 를 봐야 합니다.
- **풀이 접근**: 채택한 모형의 회귀계수 보고표를 만들어 β 열을 절댓값 크기순으로 비교합니다. 등분산 가정이 깨졌는지에 따라 유의성 판정에 쓸 표준오차(일반 OLS / HC3)도 함께 확인합니다.
- **근거(계산 결과)**: β 는 smoker **0.794**, age 0.299, bmi 0.162, children 0.047 순입니다. 흡연 여부의 영향력이 나이의 2.7배에 달합니다. 회귀계수 B 로 보면 흡연자는 비흡연자보다 연간 23,810.4 달러를 더 부담하는 셈이고, 네 변수 모두 p < .05 로 유의합니다(VIF 는 최대 1.014 로 다중공선성 문제도 없습니다).
- **실무 포인트**: β 의 순위는 영향력의 순위일 뿐 "흡연이 나이보다 2.7배 중요하다"처럼 배수로 단정하는 근거는 아닙니다. 다만 요율 산정에서 흡연 여부 할증이 가장 큰 항목이 되어야 한다는 판단에는 충분한 근거입니다.

#### 6. 채택한 모형을 그대로 요율 산정에 쓰기 전에 네 가지 회귀분석 가정을 점검했습니다. 이 중 유일하게 충족된 가정은 무엇인가요?

- **정답**: `독립성`
- **복습 개념**: 회귀모형의 네 가정(선형성·정규성·등분산성·독립성) 진단입니다. 모형이 유의하고 설명력이 높아도 가정이 깨졌다면 계수의 표준오차와 유의성 판정을 신뢰할 수 없습니다.
- **풀이 접근**: 채택한 모형의 잔차로 네 가정을 차례로 검정합니다. 선형성·정규성·등분산성은 귀무가설이 "가정을 만족한다"이므로 p 값이 0.05보다 작으면 위배이고, 독립성은 검정통계량이 2 근처인지로 판정합니다.
- **근거(계산 결과)**: 선형성 141.672 (p = 0.000) → 위배, 정규성 0.163 (p = 0.000) → 위배, 등분산성 F = 31.929 (p = 0.000) → 위배, 독립성 2.088 → 만족입니다. 즉 충족된 가정은 **독립성** 하나뿐입니다.
- **결론**: 당신이 팀장에게 보고할 내용은 이렇습니다. 전처리를 더 한 데이터가 아니라 **0_기준선**이 달러 기준 오차가 가장 작아(RMSE 6,058.64 달러) 채택되었고, 흡연 여부가 가장 큰 요율 요인이라는 것. 다만 등분산성이 깨져 유의성 판정은 HC3 로버스트 표준오차로 보고해야 하고, 선형성·정규성까지 위배되었으니 이 모형을 최종 요율표로 확정하기 전에 곡선 관계를 담을 항을 추가하거나 고액 계약 구간을 따로 모형화하는 개선이 필요하다는 것까지 함께 전해야 합니다.